<a href="https://colab.research.google.com/github/Abhi00728/ML-PROJECT/blob/pranay-eda-setup/01_eda_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, glob
import kagglehub

# Downloads only accepted_2007_to_2018Q4.csv.gz (skips the rejected file)
downloaded = kagglehub.dataset_download(
    "wordsforthewise/lending-club",
    path="accepted_2007_to_2018Q4.csv.gz"
)
print("Downloaded path:", downloaded)

# Handle both direct file and nested folder extraction cleanly
if os.path.isdir(downloaded):
    csv_path = glob.glob(os.path.join(downloaded, "**", "*accepted*"), recursive=True)[0]
else:
    csv_path = downloaded

print("Using file:", csv_path)

Using Colab cache for faster access to the 'lending-club' dataset.
Downloaded path: /kaggle/input/lending-club/accepted_2007_to_2018Q4.csv.gz
Using file: /kaggle/input/lending-club/accepted_2007_to_2018Q4.csv.gz


In [2]:
import pandas as pd

# Safeguard: if Cell 1 matched the folder instead of the file inside it, grab the actual file
if os.path.isdir(csv_path):
    inner_files = [
        f for f in glob.glob(os.path.join(csv_path, "**", "*"), recursive=True)
        if os.path.isfile(f)
    ]
    csv_path = inner_files[0]

# Check gzip magic bytes so pandas uses the right decompression
with open(csv_path, "rb") as f:
    compression = "gzip" if f.read(2) == b"\x1f\x8b" else None

print("Resolved file:", csv_path)
print("File size (MB):", round(os.path.getsize(csv_path) / (1024**2), 1))
print("Compression:", compression)

# Load a 50,000-row sample safely (low RAM)
sample = pd.read_csv(csv_path, compression=compression, nrows=50_000, low_memory=False)
print("Sample shape:", sample.shape)
display(sample.head(5))

# Full 151-column audit table
audit = pd.DataFrame({
    "dtype": sample.dtypes.astype(str),
    "missing_pct": (sample.isna().mean() * 100).round(2),
    "n_unique": sample.nunique(),
    "example": sample.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None)
})

pd.set_option("display.max_rows", 160)
display(audit)

Resolved file: /kaggle/input/lending-club/accepted_2007_to_2018Q4.csv.gz
File size (MB): 374.4
Compression: gzip
Sample shape: (50000, 151)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


,dtype,missing_pct,n_unique,example
id,int64,0.00,50000,68407277
member_id,float64,100.00,0,None
loan_amnt,float64,0.00,1293,3600.0
funded_amnt,float64,0.00,1293,3600.0
funded_amnt_inv,float64,0.00,1316,3600.0
term,object,0.00,2,36 months
int_rate,float64,0.00,70,13.99
installment,float64,0.00,11548,123.03
grade,object,0.00,7,C
sub_grade,object,0.00,35,C4


In [3]:
meta = pd.read_csv(
    csv_path,
    compression=compression,
    usecols=["issue_d", "loan_status"],
    low_memory=False
)

print("Total rows in dataset:", f"{len(meta):,}")
print("\n--- Full loan_status counts ---")
print(meta["loan_status"].value_counts(dropna=False))

meta["issue_year"] = pd.to_datetime(meta["issue_d"], format="%b-%Y", errors="coerce").dt.year
resolved = meta[meta["loan_status"].isin(["Fully Paid", "Charged Off", "Default"])]

print("\n--- Resolved loans by issue year ---")
ct = pd.crosstab(resolved["issue_year"], resolved["loan_status"])
ct["Total"] = ct.sum(axis=1)
ct["Default_Rate_%"] = (
    (ct.get("Charged Off", 0) + ct.get("Default", 0)) / ct["Total"] * 100
).round(2)
display(ct)

Total rows in dataset: 2,260,701

--- Full loan_status counts ---
loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
NaN                                                         33
Name: count, dtype: int64

--- Resolved loans by issue year ---


loan_status,Charged Off,Default,Fully Paid,Total,Default_Rate_%
issue_year,,,,,
2007.0,45,0,206,251,17.93
2008.0,247,0,1315,1562,15.81
2009.0,594,0,4122,4716,12.60
2010.0,1487,0,10049,11536,12.89
2011.0,3297,0,18424,21721,15.18
2012.0,8644,0,44723,53367,16.20
2013.0,21024,0,113780,134804,15.60
2014.0,41161,1,181941,223103,18.45
2015.0,75803,1,299742,375546,20.19


In [5]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/ML_Loan_Default_Project"
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder ready:", PROJECT_DIR)

Mounted at /content/drive
Project folder ready: /content/drive/MyDrive/ML_Loan_Default_Project


In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Explicit post-origination leakage columns (recorded AFTER loan approval -> MUST DROP)
LEAKAGE_COLS = [
    "funded_amnt", "funded_amnt_inv", "pymnt_plan", "out_prncp", "out_prncp_inv",
    "total_pymnt", "total_pymnt_inv", "total_rec_prncp", "total_rec_int",
    "total_rec_late_fee", "recoveries", "collection_recovery_fee",
    "last_pymnt_d", "last_pymnt_amnt", "next_pymnt_d", "last_credit_pull_d",
    "last_fico_range_high", "last_fico_range_low", "debt_settlement_flag",
    "debt_settlement_flag_date", "settlement_status", "settlement_date",
    "settlement_amount", "settlement_percentage", "settlement_term",
    "hardship_flag", "hardship_type", "hardship_reason", "hardship_status",
    "deferral_term", "hardship_amount", "hardship_start_date", "hardship_end_date",
    "payment_plan_start_date", "hardship_length", "hardship_dpd",
    "hardship_loan_status", "orig_projected_additional_accrued_interest",
    "hardship_payoff_balance_amount", "hardship_last_payment_amount",
    "disbursement_method"
]

# 2. IDs, 100%-empty, constant, or high-cardinality free-text columns
META_DROP_COLS = [
    "id", "member_id", "url", "desc", "emp_title", "title", "zip_code", "policy_code"
]

all_cols = pd.read_csv(csv_path, compression=compression, nrows=0).columns.tolist()
drop_set = set(LEAKAGE_COLS + META_DROP_COLS)
use_cols = [c for c in all_cols if c not in drop_set]

print(f"Total raw columns: {len(all_cols)}")
print(f"Dropped leakage + metadata columns: {len(all_cols) - len(use_cols)}")

# 3. Read in chunks & filter to resolved 2015-2016 loans
chunks = []
for chunk in pd.read_csv(
    csv_path, compression=compression, usecols=use_cols,
    chunksize=200_000, low_memory=False
):
    chunk = chunk[chunk["loan_status"].isin(["Fully Paid", "Charged Off", "Default"])].copy()
    chunk["issue_dt"] = pd.to_datetime(chunk["issue_d"], format="%b-%Y", errors="coerce")
    chunk["issue_year"] = chunk["issue_dt"].dt.year
    chunk = chunk[chunk["issue_year"].isin([2015, 2016])]
    if len(chunk) > 0:
        chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
print(f"Filtered 2015-2016 resolved loans: {df.shape}")

# 4. Create binary target: 1 = Default / Charged Off, 0 = Fully Paid
df["target"] = df["loan_status"].isin(["Charged Off", "Default"]).astype("int8")
df.drop(columns=["loan_status"], inplace=True)

# 5. Parse messy text/date columns into clean numeric features
df["term_months"] = df["term"].str.extract(r"(\d+)").astype(float)
df["emp_length_years"] = (
    df["emp_length"]
    .replace({"< 1 year": "0", "10+ years": "10"})
    .str.extract(r"(\d+)")
    .astype(float)
)
earliest_dt = pd.to_datetime(df["earliest_cr_line"], format="%b-%Y", errors="coerce")
df["credit_history_months"] = ((df["issue_dt"] - earliest_dt).dt.days / 30.44).round(1)

df.drop(columns=["term", "emp_length", "earliest_cr_line", "issue_d", "issue_dt"], inplace=True)

# 6. Handle high-missingness columns thoughtfully:
# Keep informative delinquency recency columns (missing = never delinquent), drop >70% missing joint-loan cols
INFORMATIVE_MISSING_COLS = {
    "mths_since_last_delinq", "mths_since_last_record",
    "mths_since_last_major_derog", "mths_since_recent_bc_dlq",
    "mths_since_recent_revol_delinq"
}
missing_frac = df.isna().mean()
high_miss_drop = [
    c for c in df.columns
    if missing_frac[c] > 0.70 and c not in INFORMATIVE_MISSING_COLS
]
df.drop(columns=high_miss_drop, inplace=True)

# Add explicit missingness indicator flags for informative delinquency columns
for col in INFORMATIVE_MISSING_COLS:
    if col in df.columns:
        df[f"{col}_is_missing"] = df[col].isna().astype("int8")

# 7. Stratified sample of 200,000 borrowers (preserves year + default ratio)
strat_key = df["issue_year"].astype(str) + "_" + df["target"].astype(str)
df_clean, _ = train_test_split(
    df, train_size=200_000, stratify=strat_key, random_state=42
)
df_clean = df_clean.reset_index(drop=True)

# 8. Save to Google Drive as Parquet (fast ~2s reload for the whole team)
out_path = os.path.join(PROJECT_DIR, "lending_clean.parquet")
df_clean.to_parquet(out_path, index=False)

feature_count = df_clean.shape[1] - 2  # excluding target and issue_year
print("\n=== CLEAN DATASET SAVED ===")
print("Saved to:", out_path)
print("Final shape:", df_clean.shape)
print(f"Clean features (excluding target & issue_year): {feature_count} (Required: >= 50)")
print("Default rate (%):", round(df_clean["target"].mean() * 100, 2))
print("Rows by year:\n", df_clean["issue_year"].value_counts().sort_index())

Total raw columns: 151
Dropped leakage + metadata columns: 49
Filtered 2015-2016 resolved loans: (668651, 104)

=== CLEAN DATASET SAVED ===
Saved to: /content/drive/MyDrive/ML_Loan_Default_Project/lending_clean.parquet
Final shape: (200000, 91)
Clean features (excluding target & issue_year): 89 (Required: >= 50)
Default rate (%): 21.54
Rows by year:
 issue_year
2015    112329
2016     87671
Name: count, dtype: int64
